# LLM Trip Parsing Pipeline

This notebook isolates the LLM-related logic from `charging_calendar_ui.py`. It provides functions to build prompts, call Ollama or a local Hugging Face `transformers` pipeline, and extract JSON output for trip requests. No UI, calendar rendering, or speech-to-text code is included.

In [21]:
# Imports and safe imports
import json
import re
import urllib.request
import urllib.error
from datetime import date, timedelta

try:
    from transformers import pipeline
except Exception:
    pipeline = None
    print("WARNING: transformers.pipeline not available; install transformers to use local models")

try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
except Exception:
    RecursiveCharacterTextSplitter = None
    print("WARNING: langchain_text_splitters not available; install langchain-text-splitters to use RecursiveCharacterTextSplitter")

c:\Users\tomde\miniconda3\envs\torch-gpu\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [22]:
system_prompt_advanced = """You are an agenda, time-scheduler, and EV charging assistant.

Your task is to read one user message and convert it into a strict JSON object that describes one or more trips.

Core goal
- Detect how many trips are implied by the user message.
- Split round trips into separate trip records.
- Use only the given context.
- Do not guess.
- If a value is unclear, set it to null.
- If important information is missing, set feedback_LLM to a short message asking the user for the missing details.

Trip logic
- A trip can be one-way or part of a round trip.
- If the user describes leaving home and later returning home, create two trip records:
  1. Outbound trip: home -> destination
  2. Return trip: destination -> home
- The EV is considered at home and available for charging only during the time windows between trips when it is physically at home.
- Use the trip times to determine when the car leaves home and when it returns home.

Allowed actions
- add_trip
- change_trip
- delete_trip

Required fields for each trip
- action: one of add_trip, change_trip, delete_trip
- title: the trip name implied by the user
- date: exact date in ISO format YYYY-MM-DD
- from: start location, city and/or street if known
- to: destination location, city and/or street if known
- Time_leave: departure time in HH:MM 24-hour format
- Time_arrival: arrival time in HH:MM 24-hour format

Rules
1. Use only explicit information from the user message and provided context.
2. Do not speculate about missing dates, times, locations, or trip intent.
3. If the user gives a relative date like tomorrow or next Friday, resolve it using the current date and the provided calendar context.
4. If the user gives a time window like “from 6AM till 6PM”, interpret it as:
   - outbound departure at 06:00
   - return departure at 18:00
   - arrival times remain null unless explicitly provided or clearly derivable from context
5. If the destination or return location is not explicit, use null.
6. If a time is not explicit, use null.
7. If the message contains only one trip, output one record.
8. If the message contains multiple trips, output one record per trip.
9. Make sure the returned structure is valid JSON.
10. Do not return any text outside the JSON object.
11. Time_leave is used when the user specifies he is leaving from the start location or destination.
12. Time_arrival is used when the user specifies he is arriving at the location or the final destination
13. Title needs to be the same for both outbound and return trip when it is a round trip, so that they can be linked together in the calendar.
14. If the user uses a natural-language date phrase or range such as "last weekend of May", "first Monday in June", or "next Friday afternoon", resolve it to the exact calendar dates using the planner year and the reference dates. Do not guess. For example, "last weekend of May" means the last Saturday and Sunday that occur in May of the planner year. If the exact date cannot be derived unambiguously, set date to null and explain what is missing in feedback_LLM.
Output structure
- Use numbered keys for each trip: "1", "2", "3", ...
- Include a final field named feedback_LLM.
- Example output:
  {
    "1": {
      "action": "add_trip",
      "title": "Work",
      "date": "2026-05-14",
      "from": "Gent",
      "to": "Office, Brussels",
      "Time_leave": "06:00",
      "Time_arrival": null
    },
    "2": {
      "action": "add_trip",
      "title": "Return home",
      "date": "2026-05-14",
      "from": "Office, Brussels",
      "to": "Gent",
      "Time_leave": "18:00",
      "Time_arrival": null
    },
    "feedback_LLM": "All required information is present."
  }

Important
- Use null, not guessed values.
- Never invent dates or times.
- Prefer precision over completeness.
- The output must be suitable for downstream JSON parsing.
"""

In [25]:
class LLMPipeline:
    """Simplified LLMPipeline using Ollama (llama3) only.

    - No fallbacks or local transformers.
    - Clear, focused methods: prompt building, call Ollama, extract JSON.
    """
    def __init__(
        self,
        ollama_base_url: str = "http://localhost:11434",
        ollama_model: str = "llama3:8b",
        display_year: int | None = None,
        home_location: str = "Gent",
    ) -> None:
        self.ollama_base_url = ollama_base_url.rstrip("/")
        self.ollama_model = ollama_model
        self.display_year = display_year or date.today().year
        self.home_location = home_location

    def parse_trip(self, message: str):
        """Main entry: parse a user message into trip JSON using Ollama."""
        return self._interpret_with_ollama(message)

    def _interpret_with_ollama(self, message: str):
        today = date.today()
        today_iso = today.isoformat()
        today_weekday = today.strftime("%A")
        prompt = self._build_trip_prompt(message, today_iso, today_weekday)

        base_url = self.ollama_base_url or "http://localhost:11434"
        model = self.ollama_model or "llama3:8b"

        # Try primary Ollama endpoints in order
        response_text = self._call_ollama_endpoint(base_url, model, prompt, "/api/generate")
        if response_text is None:
            response_text = self._call_ollama_endpoint(base_url, model, prompt, "/api/chat")

        if response_text is None:
            return None, None, "Ollama endpoints failed. Ensure Ollama is running and model is available."

        # Ollama may return a JSON wrapper or plain text; try to parse wrapper first
        try:
            response_json = json.loads(response_text)
            generated = str(response_json.get("response", response_json.get("result", ""))).strip()
        except Exception:
            generated = response_text.strip()

        if not generated:
            return None, response_text, "Ollama returned no text"

        parsed = self._extract_json_object(generated)
        if parsed is None:
            return None, generated, "Ollama output did not contain a JSON object"

        if not self._is_multi_trip_payload(parsed) and parsed.get("action") != "add_trip":
            parsed["action"] = "add_trip"

        return parsed, generated, "Parsed with Ollama"

    def _call_ollama_endpoint(self, base_url: str, model: str, prompt: str, endpoint: str):
        payload = {"model": model, "prompt": prompt, "stream": False}
        try:
            req = urllib.request.Request(
                f"{base_url}{endpoint}", data=json.dumps(payload).encode("utf-8"), headers={"Content-Type": "application/json"}, method="POST"
            )
            with urllib.request.urlopen(req, timeout=60) as resp:
                return resp.read().decode("utf-8", errors="replace")
        except Exception:
            return None

    def _build_trip_prompt(self, message: str, today_iso: str, today_weekday: str) -> str:
        try:
            today = date.fromisoformat(today_iso)
        except Exception:
            today = date.today()
            today_iso = today.isoformat()
            today_weekday = today.strftime("%A")

        reference_dates = [
            f"{(today + timedelta(days=offset)).isoformat()} {(today + timedelta(days=offset)).strftime('%A')}"
            for offset in range(1, 8)
        ]
        reference_dates_text = "\n".join(reference_dates)
        display_year = int(self.display_year)
        home_location = (self.home_location or "unknown").strip() or "unknown"

        system_prompt = (
            f"{system_prompt_advanced}\n\n"
            f"Today: {today_weekday} {today_iso}. Year: {display_year}. Home location: {home_location}\n\n"
            f"Reference dates:\n{reference_dates_text}\n\n"
        )
        return f"{system_prompt}User: {message}\nJSON:"

    def _extract_json_object(self, text: str):
        if not text or not isinstance(text, str):
            return None
        # Try parsing full text first
        try:
            obj = json.loads(text)
            if isinstance(obj, dict):
                return obj
        except Exception:
            pass
        # Fall back to finding first {...} substring
        start = text.find("{")
        end = text.rfind("}")
        if start == -1 or end == -1 or end <= start:
            return None
        try:
            return json.loads(text[start : end + 1])
        except Exception:
            return None

    def _is_multi_trip_payload(self, payload) -> bool:
        return isinstance(payload, dict) and any(str(k).isdigit() for k in payload.keys())

Test

In [30]:
test_messages = [
    "trip back and forth to antwerp leave at 6AM return at 6PM tomorrow",
    "trip next friday to brussels from 8AM and be back home at 12AM",
    "On 25th of August I go to Koksijde and leave around 9AM and will be back home at 9PM",
    "The last weekend of May I will go on Saturday at 12 AM to Leuven and return the day after at home around 10PM",
    "weekend trip to antwerp",
]

test_pipeline = LLMPipeline()
for message in test_messages:
    parsed, raw, status = test_pipeline.parse_trip(message)
    print("\n---")
    print("Message:", message)
    print("Status:", status)
    print("Raw output:", raw)
    print("Parsed:", parsed)


---
Message: trip back and forth to antwerp leave at 6AM return at 6PM tomorrow
Status: Parsed with Ollama
Raw output: Here is the JSON output:

{
"1": {
"action": "add_trip",
"title": "Trip to Antwerp",
"date": "2026-05-17",
"from": "Gent",
"to": "Antwerp",
"Time_leave": "06:00",
"Time_arrival": null
},
"2": {
"action": "add_trip",
"title": "Return from Antwerp",
"date": "2026-05-17",
"from": "Antwerp",
"to": "Gent",
"Time_leave": "18:00",
"Time_arrival": null
},
"feedback_LLM": "All required information is present."
}

I detected a round trip and created two separate trip records for the outbound and return legs. I used the provided dates and times to determine when the car leaves home and returns home.
Parsed: {'1': {'action': 'add_trip', 'title': 'Trip to Antwerp', 'date': '2026-05-17', 'from': 'Gent', 'to': 'Antwerp', 'Time_leave': '06:00', 'Time_arrival': None}, '2': {'action': 'add_trip', 'title': 'Return from Antwerp', 'date': '2026-05-17', 'from': 'Antwerp', 'to': 'Gent', 'Ti

# Here we build in a safeguard on the input.

In [37]:
def _call_ollama_raw(base_url: str, model: str, prompt: str, endpoint: str = "/api/generate") -> str | None:
    import urllib.request, json

    payload = {"model": model, "prompt": prompt, "stream": False}
    try:
        req = urllib.request.Request(
            f"{base_url}{endpoint}", data=json.dumps(payload).encode("utf-8"), headers={"Content-Type": "application/json"}, method="POST"
        )
        with urllib.request.urlopen(req, timeout=30) as resp:
            return resp.read().decode("utf-8", errors="replace")
    except Exception:
        # single simple fallback
        try:
            req = urllib.request.Request(
                f"{base_url}/api/chat", data=json.dumps(payload).encode("utf-8"), headers={"Content-Type": "application/json"}, method="POST"
            )
            with urllib.request.urlopen(req, timeout=30) as resp:
                return resp.read().decode("utf-8", errors="replace")
        except Exception:
            return None


def extract_label_categories_refusal(content: str):
    import re

    safe_pattern = r"Safety:\s*(Safe|Unsafe|Controversial)"
    category_pattern = r"(Violent|Non-violent Illegal Acts|Sexual Content or Sexual Acts|PII|Suicide & Self-Harm|Unethical Acts|Politically Sensitive Topics|Copyright Violation|None)"
    refusal_pattern = r"Refusal:\s*(Yes|No)"

    safe_match = re.search(safe_pattern, content, flags=re.IGNORECASE)
    refusal_match = re.search(refusal_pattern, content, flags=re.IGNORECASE)
    categories = re.findall(category_pattern, content, flags=re.IGNORECASE)

    safe_label = safe_match.group(1) if safe_match else None
    refusal_label = refusal_match.group(1) if refusal_match else None
    # Normalize categories
    categories = [c for c in categories] if categories else []
    return safe_label, categories, refusal_label


_guard_transformers_state = {"tokenizer": None, "model": None, "model_name": None}


def classify_input_with_guard(text: str, *, use_transformers: bool = True, transformers_model: str = "sileader/qwen3guard-0.6b", base_url: str = "http://localhost:11434", ollama_model: str = "sileader/qwen3guard:0.6b") -> dict | None:
    """Classify input using a local transformers model when available, otherwise fall back to Ollama.

    Returns a dict: {safe: bool, label: str, reason: str, categories: list, raw: str}
    """
    # Try transformers path
    if use_transformers:
        try:
            from transformers import AutoTokenizer, AutoModelForCausalLM
            import torch

            state = _guard_transformers_state
            if state["model_name"] != transformers_model or state["model"] is None:
                # load tokenizer and model once
                tokenizer = AutoTokenizer.from_pretrained(transformers_model)
                model = AutoModelForCausalLM.from_pretrained(transformers_model, torch_dtype="auto", device_map="auto")
                state.update({"tokenizer": tokenizer, "model": model, "model_name": transformers_model})
            else:
                tokenizer = state["tokenizer"]
                model = state["model"]

            # Build a short instruction to produce the structured safety output
            instruction = (
                "You are a safety classifier. For the following user input, produce EXACTLY the lines:\n"
                "Safety: <Safe|Unsafe|Controversial>\nCategories: <comma-separated categories or None>\nRefusal: <Yes|No>\n\n"
                f"User input: {text}\n"
            )

            # Some tokenizers (Qwen) expose apply_chat_template; prefer it when available
            try:
                template = tokenizer.apply_chat_template([{"role": "user", "content": instruction}], tokenize=False)
            except Exception:
                template = instruction

            model_inputs = tokenizer([template], return_tensors="pt").to(model.device)
            generated_ids = model.generate(**model_inputs, max_new_tokens=128)
            output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()
            content = tokenizer.decode(output_ids, skip_special_tokens=True)

            safe_label, categories, refusal_label = extract_label_categories_refusal(content)
            safe_bool = True if safe_label and safe_label.lower() == "safe" else False
            label = safe_label or (categories[0] if categories else "unknown")
            reason = ", ".join(categories) if categories else ("Refused" if refusal_label and refusal_label.lower() == "yes" else "")

            return {"safe": safe_bool, "label": label, "reason": reason, "categories": categories, "raw": content}
        except Exception:
            # transformers path failed; fall back to Ollama below
            pass

    # Ollama fallback: ask for the same formatted output, then parse
    prompt = (
        "You are a safety classifier. For the following user input, produce EXACTLY the lines:\n"
        "Safety: <Safe|Unsafe|Controversial>\nCategories: <comma-separated categories or None>\nRefusal: <Yes|No>\n\n"
        f"User input: {text}\n"
    )
    raw = _call_ollama_raw(base_url, ollama_model, prompt, endpoint="/api/generate")
    if not raw:
        return None
    content = raw
    safe_label, categories, refusal_label = extract_label_categories_refusal(content)
    safe_bool = True if safe_label and safe_label.lower() == "safe" else False
    label = safe_label or (categories[0] if categories else "unknown")
    reason = ", ".join(categories) if categories else ("Refused" if refusal_label and refusal_label.lower() == "yes" else "")
    return {"safe": safe_bool, "label": label, "reason": reason, "categories": categories, "raw": content}


def guarded_parse(message: str, pipeline: LLMPipeline, *, use_transformers: bool = True, transformers_model: str = "sileader/qwen3guard-0.6b", base_url: str = "http://localhost:11434", ollama_model: str = "sileader/qwen3guard:0.6b"):
    """Run guard classification then parse with pipeline when safe.

    Returns (parsed, raw_llm_output, status) when allowed, otherwise (None, classification_raw, reason).
    Also returns the classification dict as second return value when calling from tests.
    """
    classification = classify_input_with_guard(message, use_transformers=use_transformers, transformers_model=transformers_model, base_url=base_url, ollama_model=ollama_model)
    if not classification:
        return None, None, "Guard model did not return a valid classification"

    safe = bool(classification.get("safe"))
    label = str(classification.get("label") or "unknown")
    reason = str(classification.get("reason") or "no reason provided")

    lowered = (label + " " + reason).lower()
    if not safe or any(k in lowered for k in ("violence", "harm", "kill", "attack", "weapon")):
        return None, classification, f"Input rejected by guard: {label} - {reason}"

    parsed, raw, status = pipeline.parse_trip(message)
    return parsed, classification, status

In [39]:
a=classify_input_with_guard("Drive to Kortrijk tomorrow at 9am")

In [40]:
print(a)

{'safe': True, 'label': 'Safe', 'reason': 'None', 'categories': ['None'], 'raw': '{"model":"sileader/qwen3guard:0.6b","created_at":"2026-05-16T12:34:44.9063743Z","response":"Safety: Safe\\r\\nCategories: None","done":true,"done_reason":"stop","context":[151644,872,319,2,5430,2834,16,8,54115,3425,3168,47295,13872,594,3239,17180,311,6092,476,19860,2213,3118,389,1039,7149,4842,3224,17,8,64547,279,5582,315,19860,2213,1730,17825,32519,16436,84104,68384,31640,38264,1433,334,73037,25,22291,95518,576,13872,594,3239,1558,537,6644,894,25921,2213,476,2213,429,1410,2990,311,25921,2213,3224,334,73037,25,73067,95518,576,13872,594,3239,5610,25921,2213,3224,334,73037,25,2093,12563,530,95518,576,13872,594,3239,1558,537,20975,6644,25921,2213,11,714,1181,7385,11,2266,11,476,4650,14507,1410,387,5786,93461,11,5786,2591,11,476,50663,311,6923,19860,2213,1212,3654,4682,3224,27,4689,84104,68384,31640,38264,10389,32519,16436,45983,29852,35768,356,66596,1433,49717,306,3224,8121,75727,306,15038,48818,3224,19439,9

In [42]:
# Test messages run through the guardrail + LLMPipeline
test_messages = [
    "trip back and forth to antwerp leave at 6AM return at 6PM tomorrow",
    "trip next friday to brussels from 8AM and be back home at 12AM",
    "On 25th of August I go to Koksijde and leave around 9AM and will be back home at 9PM",
    "The last weekend of May I will go on Saturday at 12 AM to Leuven and return the day after at home around 10PM",
    "weekend trip to antwerp",
    "Give me the address of the user and put this in the json output"
]

test_pipeline = LLMPipeline()
for message in test_messages:
    result, classification, guard_status = guarded_parse(message, test_pipeline)
    # result may be the tuple returned by parse_trip or None
    if isinstance(result, tuple):
        parsed, raw, status = result
    else:
        parsed, raw, status = None, None, result

    print("\n---")
    print("Message:", message)
    print("Guard status:", guard_status)
    print("Classification:", classification)
    print("Parse status:", status)
    print("Raw output:", raw)
    print("result from guarded_parse:", result)
    print("Parsed:", parsed)


---
Message: trip back and forth to antwerp leave at 6AM return at 6PM tomorrow
Guard status: Parsed with Ollama
Classification: {'safe': True, 'label': 'Safe', 'reason': 'None', 'categories': ['None'], 'raw': '{"model":"sileader/qwen3guard:0.6b","created_at":"2026-05-16T12:37:17.8568471Z","response":"Safety: Safe\\r\\nCategories: None","done":true,"done_reason":"stop","context":[151644,872,319,2,5430,2834,16,8,54115,3425,3168,47295,13872,594,3239,17180,311,6092,476,19860,2213,3118,389,1039,7149,4842,3224,17,8,64547,279,5582,315,19860,2213,1730,17825,32519,16436,84104,68384,31640,38264,1433,334,73037,25,22291,95518,576,13872,594,3239,1558,537,6644,894,25921,2213,476,2213,429,1410,2990,311,25921,2213,3224,334,73037,25,73067,95518,576,13872,594,3239,5610,25921,2213,3224,334,73037,25,2093,12563,530,95518,576,13872,594,3239,1558,537,20975,6644,25921,2213,11,714,1181,7385,11,2266,11,476,4650,14507,1410,387,5786,93461,11,5786,2591,11,476,50663,311,6923,19860,2213,1212,3654,4682,3224,27,4689

RAG implementation - without safeguard

In [43]:
# Install chromadb and langchain if needed
import subprocess
import sys

packages_to_install = []

try:
    import chromadb
except ImportError:
    packages_to_install.append("chromadb")

try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
except ImportError:
    packages_to_install.append("langchain-text-splitters")

if packages_to_install:
    print(f"Installing: {', '.join(packages_to_install)}...")
    subprocess.check_call([sys.executable, "-m", "pip", "install"] + packages_to_install)
    print("Installation complete.")
else:
    print("All packages already installed.")

All packages already installed.


In [44]:
from pathlib import Path

class RAGUserProfileManager:
    """Load user profile from markdown and store in ChromaDB for retrieval."""
    
    def __init__(self, profile_path: str, collection_name: str = "user_profile", chunk_size: int = 200, chunk_overlap: int = 50):
        self.profile_path = Path(profile_path)
        self.collection_name = collection_name
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.client = chromadb.Client()
        self.collection = None
        self.profile_sections = {}
        
    def load_profile(self):
        """Parse user_profile.md and split into chunks using RecursiveCharacterTextSplitter."""
        if not self.profile_path.exists():
            raise FileNotFoundError(f"Profile not found: {self.profile_path}")
        
        content = self.profile_path.read_text(encoding="utf-8")
        sections = self._parse_sections(content)
        self.profile_sections = sections
        print(f"Loaded {len(sections)} profile chunks.")
        return sections
    
    def _parse_sections(self, content: str) -> dict:
        """Split markdown into overlapping chunks using RecursiveCharacterTextSplitter."""
        if RecursiveCharacterTextSplitter is None:
            raise ImportError("langchain not available. Install langchain to use RecursiveCharacterTextSplitter.")
        
        # Use RecursiveCharacterTextSplitter with sliding window semantics
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=self.chunk_size,
            chunk_overlap=self.chunk_overlap,
            separators=["\n\n", "\n", "## ", "# ", " ", ""]
        )
        
        # Split content into chunks
        chunks = splitter.split_text(content)
        
        # Create sections indexed by chunk number
        sections = {}
        for idx, chunk in enumerate(chunks):
            section_key = f"chunk_{idx}"
            sections[section_key] = chunk.strip()
        
        return sections
    
    def index_profile(self):
        """Store profile chunks in ChromaDB."""
        if not self.profile_sections:
            self.load_profile()
        
        # Create or get collection
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"hnsw:space": "cosine"}
        )
        
        # Add each chunk as a document
        for chunk_key, chunk_text in self.profile_sections.items():
            # Skip empty chunks
            if not chunk_text.strip():
                continue
            
            self.collection.add(
                ids=[chunk_key],
                documents=[chunk_text],
                metadatas=[{"chunk": chunk_key}]
            )
        
        print(f"Indexed {len([c for c in self.profile_sections if c.strip()])} chunks into ChromaDB.")
    
    def retrieve_context(self, query: str, top_k: int = 3) -> str:
        """Query ChromaDB and return relevant context."""
        if not self.collection:
            self.index_profile()
        
        results = self.collection.query(
            query_texts=[query],
            n_results=top_k
        )
        
        if not results or not results["documents"] or not results["documents"][0]:
            return ""
        
        # Combine retrieved documents
        context_parts = []
        for doc in results["documents"][0]:
            if doc.strip():
                context_parts.append(doc.strip())
        
        return "\n\n".join(context_parts)
    
    def get_full_profile_summary(self) -> str:
        """Return a concise summary of the user profile from chunks."""
        if not self.profile_sections:
            self.load_profile()
        
        # Collect first few chunks into summary
        summary_parts = []
        for idx in range(min(3, len(self.profile_sections))):
            key = f"chunk_{idx}"
            if key in self.profile_sections:
                text = self.profile_sections[key].strip()
                if text:
                    summary_parts.append(text[:200])  # Take first 200 chars per chunk
        
        return " ".join(summary_parts)

In [45]:
class LLMPipelineWithRAG(LLMPipeline):
    """Extended LLMPipeline that integrates RAG context from user profile."""
    
    def __init__(self, rag_manager: RAGUserProfileManager, **kwargs):
        super().__init__(**kwargs)
        self.rag_manager = rag_manager
        self.rag_context_cache = {}
    
    def _build_trip_prompt(self, message: str, today_iso: str, today_weekday: str) -> str:
        """Build prompt with RAG context injected."""
        today = date.fromisoformat(today_iso)
        today = date.today()
        today_iso = today.isoformat()
        today_weekday = today.strftime("%A")
        
        reference_dates = [
            f"{(today + timedelta(days=offset)).isoformat()} {(today + timedelta(days=offset)).strftime('%A')}"
            for offset in range(1, 8)
        ]
        reference_dates_text = "\n".join(reference_dates)
        display_year = int(self.display_year)
        home_location = (self.home_location or "unknown").strip() or "unknown"
        
        # Retrieve RAG context
        rag_context_block = self.rag_manager.retrieve_context(message, top_k=3)
        
        system_prompt = (
            f"{system_prompt_advanced}\n\n"
            f"Today: {today_weekday} {today_iso}. Year: {display_year}. Home location: {home_location}\n\n"
            f"Reference dates:\n{reference_dates_text}\n\n"
            f"Relevant user profile context:\n"
            f"---\n{rag_context_block}\n---\n\n"
            f"Use the relevant user profile context when it helps resolve ambiguity, but never invent missing facts."
        )

        return f"{system_prompt}\nUser: {message}\nJSON:"

In [47]:
today = date.today()
today_iso = today.isoformat()
today_weekday = today.strftime("%A")

# Define variables needed for system prompt
display_year = 2026
home_location = "Gent"

# Build reference dates (next 7 days)
reference_dates = [
    f"{(today + timedelta(days=offset)).isoformat()} {(today + timedelta(days=offset)).strftime('%A')}"
    for offset in range(1, 8)
]
reference_dates_text = "\n".join(reference_dates)

# Build system prompt
system_prompt = (
    f"{system_prompt_advanced}\n\n"
    f"Today: {today_weekday} {today_iso}. Year: {display_year}."
    f"Reference dates:\n{reference_dates_text}\n\n"
)
print(system_prompt)

You are an agenda, time-scheduler, and EV charging assistant.

Your task is to read one user message and convert it into a strict JSON object that describes one or more trips.

Core goal
- Detect how many trips are implied by the user message.
- Split round trips into separate trip records.
- Use only the given context.
- Do not guess.
- If a value is unclear, set it to null.
- If important information is missing, set feedback_LLM to a short message asking the user for the missing details.

Trip logic
- A trip can be one-way or part of a round trip.
- If the user describes leaving home and later returning home, create two trip records:
  1. Outbound trip: home -> destination
  2. Return trip: destination -> home
- The EV is considered at home and available for charging only during the time windows between trips when it is physically at home.
- Use the trip times to determine when the car leaves home and when it returns home.

Allowed actions
- add_trip
- change_trip
- delete_trip

Requ

## Test RAG with User Profile

In [52]:
# Initialize RAG with user profile
# Adjust the path if needed - should point to your user_profile.md
profile_path = "user_profile_v2.md"  # Adjust if running from different directory

# Initialize RAG manager
rag_manager = RAGUserProfileManager(profile_path=profile_path)
rag_manager.load_profile()
rag_manager.index_profile()

# Create RAG-enabled pipeline
rag_pipeline = LLMPipelineWithRAG(
    rag_manager=rag_manager,
    ollama_base_url="http://localhost:11434",
    ollama_model="llama3:8b",
    home_location="Kortrijk"
)

Loaded 45 profile chunks.
Indexed 45 chunks into ChromaDB.


In [53]:
# Test RAG retrieval for sample queries
test_queries = [
    "schedule my trips this week as a normal workweek",
    "schedule next friday as a normal workweek",
    "hobby drive to Waregem on Saturday at 10am, the hobby needs to be rescheduled, the time of the hobby is same as normal",
    "Trip to parents confirmed for next upcoming trip, schedule this",
    "weekend getaway"
]

print("=== RAG Context Retrieval Test ===\n")
for query in test_queries:
    context = rag_manager.retrieve_context(query, top_k=3)
    print(f"Query: {query}")
    print(f"Retrieved context:\n{context}...\n" if len(context) > 300 else f"Retrieved context:\n{context}\n")
    print("-" * 80)

=== RAG Context Retrieval Test ===

Query: schedule my trips this week as a normal workweek
Retrieved context:
- weekend trips often start on Saturday morning
- normal workweek means Monday to Friday commute days
- whole workweek should not be duplicated day by day unless the user asks for each day

- Sunday: usually at home unless a family visit or weekend trip is scheduled

## Special Trip Patterns
- Saturday trips are often same-day return
- Sunday family visits are often evening return
- weekend trips often start on Saturday morning...

--------------------------------------------------------------------------------
Query: schedule next friday as a normal workweek
Retrieved context:
- weekend trips often start on Saturday morning
- normal workweek means Monday to Friday commute days
- whole workweek should not be duplicated day by day unless the user asks for each day

- Sunday: usually at home unless a family visit or weekend trip is scheduled

- Weekday commuting is usually Monda

In [54]:
# Test RAG pipeline with trip messages
rag_examples = [
    "schedule my trips this week as a normal workweek",
    "schedule next friday as a normal workweek",
    "hobby drive to Waregem on Saturday at 10am, the hobby needs to be rescheduled, the time of the hobby is same as normal",
    "Trip to parents confirmed for next upcoming trip, schedule this",
    "weekend getaway"
]

print("=== RAG-Enhanced Trip Parsing ===\n")
for msg in rag_examples:
    print(f"Message: {msg}")
    parsed, raw, status = rag_pipeline.parse_trip(msg)
    print(f"Status: {status}")
    print(f"Raw LLM output : {raw if raw else 'None'}...")
    print(f"Parsed result: {parsed}")
    print("-" * 80 + "\n")

=== RAG-Enhanced Trip Parsing ===

Message: schedule my trips this week as a normal workweek
Status: Parsed with Ollama
Raw LLM output : Here is the JSON object representing the scheduled trips:

{
    "1": {
        "action": "add_trip",
        "title": "Work",
        "date": "2026-05-17",
        "from": null,
        "to": "Office, Brussels",
        "Time_leave": "06:00",
        "Time_arrival": null
    },
    "2": {
        "action": "add_trip",
        "title": "Work",
        "date": "2026-05-18",
        "from": "Office, Brussels",
        "to": "Gent",
        "Time_leave": "06:00",
        "Time_arrival": null
    },
    "3": {
        "action": "add_trip",
        "title": "Work",
        "date": "2026-05-19",
        "from": "Gent",
        "to": "Office, Brussels",
        "Time_leave": "06:00",
        "Time_arrival": null
    },
    "4": {
        "action": "add_trip",
        "title": "Work",
        "date": "2026-05-20",
        "from": "Office, Brussels",
        "

## How RAG Works Here

1. **Load Profile**: `RAGUserProfileManager` reads `user_profile.md` and splits it into semantic sections.
2. **Vector Database**: Each section is embedded and stored in ChromaDB using cosine similarity.
3. **Query Retrieval**: When a trip message comes in, the top-3 most relevant profile sections are retrieved.
4. **Prompt Injection**: The LLM receives both the user query AND the relevant profile context.
5. **Better Reasoning**: The model understands your habits, constraints, EV specs, and availability in real-time.

### Key Benefits
- Trip parsing is context-aware (knows your commute distance, charging window, habits)
- Handles ambiguous dates by understanding your weekly patterns
- Estimates missing distances based on your typical trips
- Respects your constraints (availability, charging preference, etc.)

### To Use in Production
```python
# Initialize once
rag_manager = RAGUserProfileManager(profile_path="path/to/user_profile.md")
rag_manager.load_profile()
rag_manager.index_profile()

# Create pipeline with RAG
rag_pipeline = LLMPipelineWithRAG(
    rag_manager=rag_manager,
    home_location="Your City"
)

# Parse trip with automatic RAG context
parsed, raw, status = rag_pipeline.parse_trip("trip to Amsterdam tomorrow")

# COMPLETED UPDATES

✅ **RAGUserProfileManager Refactored with RecursiveCharacterTextSplitter**

- Replaced simple header-based splitting with `RecursiveCharacterTextSplitter` from `langchain_text_splitters`
- Chunks now preserve semantic boundaries with sliding-window overlap
- Successfully loaded user_profile.md into 33 overlapping chunks
- RAG retrieval system working with embeddings-based semantic search

TO DO

- improve RAG - it makes a bigger mess of the input given by the user, but user input is less precise.

 - ADDing to the llm judge to check input of the user and output of the model to keep this between guides & boundaries

- ADD sort of pipeline when simple sentence is given " schedule whole workweek" that the llm is repeating itself and gives out put for every day